# Notebook 13: Retrieval-Augmented Generation

## Why This Matters
RAG is the dominant pattern for knowledge-intensive LLM applications. Instead of encoding all knowledge in model weights — which requires retraining to update — you retrieve relevant documents at query time and pass them as context. The model's job shifts from "recall the answer" to "synthesize an answer from the provided documents."

This notebook builds a complete RAG pipeline from scratch and evaluates it rigorously.

## Structure
1. RAG architecture — why it works and when it doesn't (narrative)
2. Document ingestion: chunk, embed, index
3. Retrieval: dense search, sparse search (BM25), hybrid
4. Generation: constructing the prompt with retrieved context
5. Evaluation: answer correctness, context relevance, faithfulness
6. Failure modes and mitigations
7. Bridge to embedding drift detection

## What You'll Learn
- How chunking strategy affects retrieval quality
- The difference between dense (embedding) and sparse (BM25) retrieval
- How to evaluate RAG with RAGAS-style metrics
- The most common RAG failure modes and how to diagnose them


## Background

### Why RAG?

LLMs have two fundamental limitations for knowledge tasks:
1. **Stale knowledge** — training cutoffs mean the model doesn't know about recent events
2. **Hallucination** — the model generates plausible-sounding but wrong facts when it doesn't know the answer

RAG addresses both: retrieve current, authoritative documents and include them in the context. The model now has the answer in front of it — the task shifts from memory recall to reading comprehension, which models are much better at.

### The pipeline

```
Documents → Chunk → Embed → Index (vector DB)
                                        ↓
Query → Embed → ANN search → Top-k docs → Prompt → LLM → Answer
```

### What goes wrong

**Retrieval failure:** The right document isn't in the top-k. More common than generation failure — fix retrieval first.

**Context window overflow:** Too many chunks retrieved → context too long → model misses relevant parts.

**Lost in the middle:** LLMs attend more to beginning and end of context than the middle. Position matters.

**Chunk boundary issues:** The answer spans two chunks; neither chunk alone is sufficient.

**Irrelevance:** Retrieved chunks are semantically similar but don't actually answer the question.


In [ ]:
# %pip install -q chromadb sentence-transformers rank-bm25 numpy torch

In [ ]:
import re
import numpy as np
import torch
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"Ready. Device: {device}")

## 1. Document Ingestion — Chunk, Embed, Index

In [ ]:
# A small knowledge base about LLM inference (simulating a documentation corpus)
raw_documents = [
    {
        "id": "doc_flash_attn",
        "title": "Flash Attention",
        "content": """Flash attention is a memory-efficient attention algorithm developed by Tri Dao et al.
        The key insight is that the standard attention computation materializes the full N×N attention
        matrix in GPU HBM (high-bandwidth memory), requiring O(N²) memory. Flash attention avoids this
        by tiling the computation: it computes attention in blocks that fit in SRAM (on-chip cache),
        accumulating the output without ever writing the full matrix to HBM. This reduces memory from
        O(N²) to O(N) and often achieves 2-4× speedup over standard attention because HBM bandwidth
        is the bottleneck. Flash attention 2 adds better parallelism and reduced non-matmul FLOPs.
        Flash attention 3 targets Hopper GPUs with FP8 support."""
    },
    {
        "id": "doc_kv_cache",
        "title": "KV Cache",
        "content": """The KV cache is a memory optimization for autoregressive generation. During
        generation, each new token attends to all previous tokens. Without caching, this requires
        recomputing the key and value projections for every past token on every step — O(N²) total
        compute. The KV cache stores these projections so each step only computes keys and values
        for the new token, reducing generation to O(N) compute. The cost is memory: a KV cache for
        a single request at 2048 tokens with a 7B model uses approximately 512MB. This limits the
        number of concurrent requests (batch size) that fit in GPU memory. PagedAttention and
        quantized KV caches are two approaches to reduce this memory pressure."""
    },
    {
        "id": "doc_speculative",
        "title": "Speculative Decoding",
        "content": """Speculative decoding (also called speculative sampling) accelerates LLM inference
        by using a small draft model to propose multiple tokens, which the large target model then
        verifies in a single forward pass. The acceptance rate — what fraction of proposed tokens
        the target model accepts — determines the speedup. Typical acceptance rates of 70-85% yield
        2-3× throughput improvements. The draft model must be fast (much smaller than the target)
        and have similar token distributions. Common draft/target pairs: LLaMA 3 8B draft for 70B,
        or a dedicated draft model trained via distillation. Speculative decoding is most effective
        for tasks with predictable outputs (code, structured data) and less effective for creative tasks."""
    },
    {
        "id": "doc_quantization",
        "title": "Quantization",
        "content": """Quantization reduces model precision to decrease memory footprint and increase
        inference throughput. FP32 (4 bytes/param) → BF16 (2 bytes) is the standard first step.
        INT8 (1 byte) requires calibration or quantization-aware training. INT4 (0.5 bytes) typically
        uses group quantization (separate scale per 128 weights) to maintain accuracy. GGUF format
        (used by llama.cpp) supports INT2 through INT8 with various quantization schemes. On Apple
        Silicon, MPS (Metal Performance Shaders) supports BF16 and some INT8 ops. Quality degradation
        with quantization: BF16 negligible, INT8 < 0.5% perplexity increase, INT4 1-3% increase,
        INT2 significant degradation. GPTQ and AWQ are popular activation-aware quantization methods
        that minimize quality loss at INT4."""
    },
]


def chunk_document(doc: dict, chunk_size: int = 200, overlap: int = 50) -> list:
    """Split document into overlapping chunks for retrieval."""
    words = doc['content'].split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if len(chunk_words) < 20:  # skip very short tail chunks
            break
        chunks.append({
            "id":       f"{doc['id']}_chunk_{len(chunks)}",
            "doc_id":   doc['id'],
            "title":    doc['title'],
            "text":     ' '.join(chunk_words),
            "chunk_idx": len(chunks),
        })
    return chunks


all_chunks = [c for doc in raw_documents for c in chunk_document(doc, chunk_size=80, overlap=20)]
print(f"Documents: {len(raw_documents)}, Chunks: {len(all_chunks)}")
for chunk in all_chunks[:3]:
    print(f"  {chunk['id']}: {chunk['text'][:80]}...")

In [ ]:
# Index chunks in Chroma
client = chromadb.Client()
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")
collection = client.create_collection("rag_corpus", embedding_function=emb_fn,
                                       metadata={"hnsw:space": "cosine"})

collection.add(
    ids       = [c['id']    for c in all_chunks],
    documents = [c['text']  for c in all_chunks],
    metadatas = [{"doc_id": c['doc_id'], "title": c['title']} for c in all_chunks],
)
print(f"Indexed {collection.count()} chunks")

# Also build BM25 for hybrid search
tokenized_chunks = [c['text'].lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index built over {len(all_chunks)} chunks")

## 2. Retrieval — Dense, Sparse, Hybrid

In [ ]:
def dense_retrieve(query: str, n: int = 3) -> list:
    results = collection.query(query_texts=[query], n_results=n)
    return list(zip(results['documents'][0], results['metadatas'][0], results['distances'][0]))

def bm25_retrieve(query: str, n: int = 3) -> list:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:n]
    return [(all_chunks[i]['text'], {'title': all_chunks[i]['title']}, -scores[i])
            for i in top_idx if scores[i] > 0]

def hybrid_retrieve(query: str, n: int = 3, alpha: float = 0.5) -> list:
    """Reciprocal rank fusion of dense and BM25 results."""
    dense = dense_retrieve(query, n=n*2)
    sparse = bm25_retrieve(query, n=n*2)

    scores = {}
    for rank, (doc, meta, _) in enumerate(dense):
        scores[doc] = scores.get(doc, 0) + alpha * 1/(rank+60)
    for rank, (doc, meta, _) in enumerate(sparse):
        scores[doc] = scores.get(doc, 0) + (1-alpha) * 1/(rank+60)

    top = sorted(scores.items(), key=lambda x:-x[1])[:n]
    return [(doc, {}, -score) for doc, score in top]


test_query = "how does flash attention reduce memory usage"
print(f"Query: {test_query!r}")
print()
for method, fn in [("Dense", dense_retrieve), ("BM25", bm25_retrieve), ("Hybrid", hybrid_retrieve)]:
    print(f"{method} retrieval:")
    for doc, meta, dist in fn(test_query):
        print(f"  [{meta.get('title','')}] {doc[:80]}...")
    print()

## 3. Generation — Prompt Construction

In [ ]:
def build_rag_prompt(query: str, retrieved_docs: list) -> str:
    """Construct a prompt with retrieved context for the LLM."""
    context_parts = []
    for i, (doc, meta, _) in enumerate(retrieved_docs, 1):
        title = meta.get('title', f'Document {i}')
        context_parts.append(f"[{i}] {title}\n{doc}")

    context = "\n\n".join(context_parts)
    return f"""Answer the question based on the provided context. If the answer is not in the context, say so.

Context:
{context}

Question: {query}

Answer:"""


# Demo the prompt structure (we won't call a real LLM here — this runs offline)
query = "what is the memory cost of the KV cache?"
docs  = dense_retrieve(query, n=2)
prompt = build_rag_prompt(query, docs)
print(prompt)
print()
print(f"Prompt length: {len(prompt.split())} words, ~{len(prompt)//4} tokens")

## 4. RAG Evaluation

In [ ]:
# Evaluate retrieval quality before generation quality
# Retrieval is the most common failure mode — measure it first

eval_queries = [
    ("how does flash attention avoid materializing the attention matrix", "doc_flash_attn"),
    ("what limits how many requests can run simultaneously", "doc_kv_cache"),
    ("what is the acceptance rate in speculative decoding", "doc_speculative"),
    ("how much memory does INT4 quantization use per parameter", "doc_quantization"),
]

print("Retrieval evaluation (recall@k):")
print(f"{'Query':<55} {'R@1':>5} {'R@3':>5} {'R@5':>5}")
print("-" * 70)

for query, relevant_doc_id in eval_queries:
    results = dense_retrieve(query, n=5)
    retrieved_doc_ids = [meta.get('doc_id') for _, meta, _ in results]

    r1 = 1 if relevant_doc_id in retrieved_doc_ids[:1] else 0
    r3 = 1 if relevant_doc_id in retrieved_doc_ids[:3] else 0
    r5 = 1 if relevant_doc_id in retrieved_doc_ids[:5] else 0

    print(f"{query[:53]:<55} {r1:>5} {r3:>5} {r5:>5}")

print()
print("If recall@1 < 0.8, fix retrieval before tuning generation.")
print("Common fixes: better chunking, hybrid retrieval, query expansion, re-ranking.")

## 5. Common Failure Modes

```
Failure mode             | Symptom                      | Fix
-------------------------|------------------------------|--------------------------------
Wrong chunk retrieved    | Answer uses wrong information | Smaller chunks, hybrid retrieval
Chunk boundary split     | Answer split across chunks   | Larger chunks + overlap
Lost in the middle       | LLM ignores middle context   | Fewer chunks, rerank to top/bottom
Irrelevant but similar   | Topical but wrong domain     | Add metadata filters
Query-document mismatch  | Question ≠ answer style      | HyDE (hypothetical document emb.)
Context too long         | Model degrades on long context | Summarize, select top-1 or top-2
```

**HyDE (Hypothetical Document Embeddings):** Instead of embedding the query, generate a hypothetical answer to the query and embed that. Search using the hypothetical answer's embedding. Works well when query and document styles differ.


## 6. Bridge to Embedding Drift Detection

Once your RAG pipeline is live, the embedding space can drift. Documents added over time may shift the distribution. User queries evolve. The embedding model may be updated. Any of these can silently degrade retrieval quality.

Notebook 14 covers embedding drift detection: how to measure when your embedding space has shifted, statistical tests for detecting it, and how to set up monitoring so you catch degradation before it affects users.


## Summary

| Stage | Key decision | Impact |
|-------|-------------|--------|
| Chunking | Size + overlap | Recall — too small loses context, too large loses precision |
| Retrieval | Dense vs BM25 vs hybrid | Recall@k — hybrid usually best |
| Reranking | Cross-encoder rerank top-20 | Precision — large quality boost |
| Prompt | Number of chunks, position | Generation quality |
| Evaluation | Retrieval recall@k first, then answer quality | Debugging priority |

**Next:** Notebook 14 — Embedding Drift Detection. How to monitor your embedding space in production and detect when retrieval quality is silently degrading.
